In [1]:
import torch
import torch.nn as nn
import sys; sys.path.insert(0, '..')
from utils import *

import warnings
warnings.filterwarnings("ignore")

In [2]:
def get_model_params(model_name):
    d, w, mc = model_scales[model_name]
    
    def ch(base):
        return min(int(base * w), mc)
    def dep(base):
        return max(round(base * d), 1)
    
    return ch, dep

In [3]:
ch, dep = get_model_params('yolo11n')

print(ch(64))
print(ch(128))
print(ch(256))
print(ch(512))
print(ch(1024))

16
32
64
128
256


<h2><strong>Backbone</strong></h2>

In [4]:
class Backbone(nn.Module):
    def __init__(self, model_name="yolo11n"):
        super().__init__()
        
        ch, dep = get_model_params(model_name)
        
        self.layer0 = Conv(3,         ch(64),   k=3, s=2)
        self.layer1 = Conv(ch(64),    ch(128),  k=3, s=2)
        self.layer2 = C3k2(ch(128),   ch(256),  n=dep(2), c3k=False, e=0.25)
        self.layer3 = Conv(ch(256),   ch(256),  k=3, s=2)
        self.layer4 = C3k2(ch(256),   ch(512),  n=dep(2), c3k=False, e=0.25)
        self.layer5 = Conv(ch(512),   ch(512),  k=3, s=2)
        self.layer6 = C3k2(ch(512),   ch(512),  n=dep(2), c3k=True,  e=0.25)
        self.layer7 = Conv(ch(512),   ch(1024), k=3, s=2)
        self.layer8 = C3k2(ch(1024),  ch(1024), n=dep(2), c3k=True,  e=0.25)
        self.layer9 = SPPF(ch(1024),  ch(1024), k=5)
    
    def forward(self, x):
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        p3 = self.layer4(x)   
        x  = self.layer5(p3)
        p4 = self.layer6(x)   
        x  = self.layer7(p4)
        x  = self.layer8(x)
        p5 = self.layer9(x)  
        return p3, p4, p5

In [5]:
backbone = Backbone("yolo11n")
dummy = torch.zeros(1, 3, 640, 640)
p3, p4, p5 = backbone(dummy)

print(f"P3 : {p3.shape}")   
print(f"P4 : {p4.shape}")   
print(f"P5 : {p5.shape}")   

P3 : torch.Size([1, 128, 80, 80])
P4 : torch.Size([1, 128, 40, 40])
P5 : torch.Size([1, 256, 20, 20])


<h2><strong>Neck Module</strong></h2>

In [6]:
class Neck(nn.Module):
    def __init__(self, model_name="yolo11n"):
        super().__init__()
        
        ch, dep = get_model_params(model_name)
        
        # ── FPN (top-down) ────────────────────────────────────────────────
        self.layer10 = C2PSA(ch(1024), ch(1024), n=dep(2))
        self.layer11 = nn.Upsample(scale_factor=2, mode='nearest')
        self.layer12 = Concat(dimension=1)
        self.layer13 = C3k2(ch(1024) + ch(512), ch(512), n=dep(2), c3k=False, e=0.5)
        
        self.layer14 = nn.Upsample(scale_factor=2, mode='nearest')
        self.layer15 = Concat(dimension=1)
        self.layer16 = C3k2(ch(512) + ch(512), ch(256), n=dep(2), c3k=False, e=0.5)

        # ── PAN (bottom-up) ───────────────────────────────────────────────
        self.layer17 = Conv(ch(256), ch(256), k=3, s=2)
        self.layer18 = Concat(dimension=1)
        self.layer19 = C3k2(ch(256) + ch(512), ch(512), n=dep(2), c3k=False, e=0.5)
        
        self.layer20 = Conv(ch(512), ch(512), k=3, s=2)
        self.layer21 = Concat(dimension=1)
        self.layer22 = C3k2(ch(512) + ch(1024), ch(1024), n=dep(2), c3k=True, e=0.5)
    
    def forward(self, p3, p4, p5):
        # ── FPN top-down ──────────────────────────────────────────────────
        x10   = self.layer10(p5)               # C2PSA    [1, 256, 20, 20]  ← save
        x     = self.layer11(x10)              # Upsample [1, 256, 40, 40]
        x     = self.layer12([x, p4])          # Concat   [1, 384, 40, 40]
        x13   = self.layer13(x)                # C3k2     [1, 128, 40, 40]  ← save
        
        x     = self.layer14(x13)              # Upsample [1, 128, 80, 80]
        x     = self.layer15([x, p3])          # Concat   [1, 256, 80, 80]
        n3    = self.layer16(x)                # C3k2     [1,  64, 80, 80]  ← output 1

        # ── PAN bottom-up ─────────────────────────────────────────────────
        x     = self.layer17(n3)               # Conv     [1,  64, 40, 40]
        x     = self.layer18([x, x13])         # Concat   [1, 192, 40, 40]
        n4    = self.layer19(x)                # C3k2     [1, 128, 40, 40]  ← output 2
        
        x     = self.layer20(n4)               # Conv     [1, 128, 20, 20]
        x     = self.layer21([x, x10])         # Concat   [1, 384, 20, 20]  ← reuse x10
        n5    = self.layer22(x)                # C3k2     [1, 256, 20, 20]  ← output 3
        
        return n3, n4, n5

In [7]:
backbone = Backbone("yolo11n")
neck     = Neck("yolo11n")

dummy = torch.zeros(1, 3, 640, 640)
p3, p4, p5 = backbone(dummy)
n3, n4, n5 = neck(p3, p4, p5)

print(f"n3 : {n3.shape}")   # expect [1,  64, 80, 80]
print(f"n4 : {n4.shape}")   # expect [1, 128, 40, 40]
print(f"n5 : {n5.shape}")   # expect [1, 256, 20, 20]

n3 : torch.Size([1, 64, 80, 80])
n4 : torch.Size([1, 128, 40, 40])
n5 : torch.Size([1, 256, 20, 20])


<h2><strong>Detection Module</strong></h2>

In [8]:
class DetectHead(nn.Module):
    def __init__(self, model_name="yolo11n", nc=1):
        super().__init__()
        
        ch, dep = get_model_params(model_name)
        
        # these must match neck's n3, n4, n5 output channels exactly
        # layer16 → ch(256), layer19 → ch(512), layer22 → ch(1024)
        neck_channels = (ch(256), ch(512), ch(1024))
        
        self.detect = Detect(nc=nc, ch=neck_channels)
        
        # always fixed for 640x640 input regardless of model variant
        self.detect.stride = torch.tensor([8.0, 16.0, 32.0])
        
        # must be called after strides are set
        self.detect.bias_init()
    
    def forward(self, x):
        return self.detect(x)

In [9]:
# Step 1 - verify channels for all variants
print("=" * 55)
print(f"{'Model':<10} {'n3':>8} {'n4':>8} {'n5':>8}")
print("=" * 55)
for name in ["yolo11n", "yolo11s", "yolo11m", "yolo11l", "yolo11x"]:
    ch, _ = get_model_params(name)
    print(f"{name:<10} {ch(256):>8} {ch(512):>8} {ch(1024):>8}")
print("=" * 55)

# Step 2 - verify strides
print("\n--- Stride Check ---")
detecthead = DetectHead("yolo11n", nc=1)
print(f"Strides : {detecthead.detect.stride}")

# Step 3 - verify output shape
print("\n--- Output Shape Check ---")
backbone   = Backbone("yolo11n")
neck       = Neck("yolo11n")
detecthead = DetectHead("yolo11n", nc=1)

dummy          = torch.zeros(1, 3, 640, 640)
p3, p4, p5     = backbone(dummy)
n3, n4, n5     = neck(p3, p4, p5)

# inference mode
detecthead.eval()
with torch.no_grad():
    out = detecthead([n3, n4, n5])
print(f"Output  : {out.shape}")

# Step 4 - verify training mode output
print("\n--- Training Mode Check ---")
detecthead.train()
out_train = detecthead([n3, n4, n5])
print(f"Training outputs  : {len(out_train)} scales")
print(f"Scale 0 (80x80)   : {out_train[0].shape}")
print(f"Scale 1 (40x40)   : {out_train[1].shape}")
print(f"Scale 2 (20x20)   : {out_train[2].shape}")

Model            n3       n4       n5
yolo11n          64      128      256
yolo11s         128      256      512
yolo11m         256      512      512
yolo11l         256      512      512
yolo11x         384      512      512

--- Stride Check ---
Strides : tensor([ 8., 16., 32.])

--- Output Shape Check ---
Output  : torch.Size([1, 5, 8400])

--- Training Mode Check ---
Training outputs  : 3 scales
Scale 0 (80x80)   : torch.Size([1, 65, 80, 80])
Scale 1 (40x40)   : torch.Size([1, 65, 40, 40])
Scale 2 (20x20)   : torch.Size([1, 65, 20, 20])


In [10]:
# import inspect
# source = inspect.getsource(Detect.forward)
# print(source)

<h1><strong>YOLOv11</strong><h1>

In [11]:
class YOLOv11(nn.Module):
    def __init__(self, model_name="yolo11n", nc=1):
        super().__init__()
        
        self.model_name = model_name
        self.nc         = nc
        
        self.backbone   = Backbone(model_name)
        self.neck       = Neck(model_name)
        self.head       = DetectHead(model_name, nc)
    
    def forward(self, x):
        p3, p4, p5      = self.backbone(x)
        n3, n4, n5      = self.neck(p3, p4, p5)
        return self.head([n3, n4, n5])

In [12]:
try:
    model = YOLOv11("yolo11n", nc=1)
    dummy = torch.zeros(1, 3, 640, 640)

    # inference mode
    model.eval()
    with torch.no_grad():
        out = model(dummy)
    print(f"Inference output : {out.shape}")   # expect [1, 5, 8400]

    # training mode
    model.train()
    out_train = model(dummy)
    print(f"Training outputs : {len(out_train)} scales")
    print(f"   Scale 0 (80x80) : {out_train[0].shape}")
    print(f"   Scale 1 (40x40) : {out_train[1].shape}")
    print(f"   Scale 2 (20x20) : {out_train[2].shape}")

    # test all variants
    print(f"\n--- All Variants Check ---")
    for name in ["yolo11n", "yolo11s", "yolo11m", "yolo11l", "yolo11x"]:
        m = YOLOv11(name, nc=1)
        m.eval()
        with torch.no_grad():
            o = m(dummy)
        print(f"{name} → {o.shape}")

except Exception as e:
    print(f"Error  : {type(e).__name__}")
    print(f"Detail : {e}")

Inference output : torch.Size([1, 5, 8400])
Training outputs : 3 scales
   Scale 0 (80x80) : torch.Size([1, 65, 80, 80])
   Scale 1 (40x40) : torch.Size([1, 65, 40, 40])
   Scale 2 (20x20) : torch.Size([1, 65, 20, 20])

--- All Variants Check ---
yolo11n → torch.Size([1, 5, 8400])
yolo11s → torch.Size([1, 5, 8400])
yolo11m → torch.Size([1, 5, 8400])
yolo11l → torch.Size([1, 5, 8400])
yolo11x → torch.Size([1, 5, 8400])


In [13]:
def model_summary(model):
    ch, _  = get_model_params(model.model_name)
    
    print("=" * 75)
    print(f"  YOLOv11 Model Summary — {model.model_name} | nc={model.nc}")
    print("=" * 75)
    print(f"  {'Section':<12} {'Layer':<12} {'Type':<20} {'Output Shape':<25} {'Params':>10}")
    print("-" * 75)
    
    dummy  = torch.zeros(1, 3, 640, 640)
    hooks  = []
    info   = []
    
    def make_hook(name, layer_num, section):
        def hook(module, input, output):
            # handle tuple outputs
            if isinstance(output, (tuple, list)):
                shape = [list(o.shape) for o in output]
            else:
                shape = list(output.shape)
            
            params = sum(p.numel() for p in module.parameters())
            info.append({
                "section"  : section,
                "layer"    : layer_num,
                "type"     : type(module).__name__,
                "shape"    : shape,
                "params"   : params
            })
        return hook
    
    # register hooks for each layer
    layer_num = 0
    for name, module in model.named_modules():
        depth = name.count(".")
        if depth == 1:  # only top-level layers
            section = name.split(".")[0]
            h = module.register_forward_hook(make_hook(name, layer_num, section))
            hooks.append(h)
            layer_num += 1
    
    # forward pass to trigger hooks
    model.eval()
    with torch.no_grad():
        model(dummy)
    
    # remove hooks
    for h in hooks:
        h.remove()
    
    # print info
    total_params = 0
    for i in info:
        print(f"  {i['section']:<12} {str(i['layer']):<12} {i['type']:<20} {str(i['shape']):<25} {i['params']:>10,}")
        total_params += i['params']
    
    print("=" * 75)
    
    # total params
    all_params     = sum(p.numel() for p in model.parameters())
    train_params   = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    print(f"  Total Parameters       : {all_params:,}")
    print(f"  Trainable Parameters   : {train_params:,}")
    print(f"  Non-trainable Params   : {all_params - train_params:,}")
    print("=" * 75)

model = YOLOv11("yolo11n", nc=1)
model_summary(model)

  YOLOv11 Model Summary — yolo11n | nc=1
  Section      Layer        Type                 Output Shape                  Params
---------------------------------------------------------------------------
  backbone     0            Conv                 [1, 16, 320, 320]                464
  backbone     1            Conv                 [1, 32, 160, 160]              4,672
  backbone     2            C3k2                 [1, 64, 160, 160]              6,640
  backbone     3            Conv                 [1, 64, 80, 80]               36,992
  backbone     4            C3k2                 [1, 128, 80, 80]              26,080
  backbone     5            Conv                 [1, 128, 40, 40]             147,712
  backbone     6            C3k2                 [1, 128, 40, 40]              32,384
  backbone     7            Conv                 [1, 256, 20, 20]             295,424
  backbone     8            C3k2                 [1, 256, 20, 20]             128,256
  backbone     9       

In [14]:
def layer_breakdown(model_name="yolo11n"):
    ch, dep = get_model_params(model_name)
    
    layers = [
        # section      layer   type      c_in            c_out        extra
        ("Backbone",   0,  "Conv",    3,              ch(64),      "k=3, s=2, 640→320"),
        ("Backbone",   1,  "Conv",    ch(64),         ch(128),     "k=3, s=2, 320→160"),
        ("Backbone",   2,  "C3k2",    ch(128),        ch(256),     f"n={dep(2)}, c3k=False"),
        ("Backbone",   3,  "Conv",    ch(256),        ch(256),     "k=3, s=2, 160→80"),
        ("Backbone",   4,  "C3k2",    ch(256),        ch(512),     f"n={dep(2)}, c3k=False ← P3"),
        ("Backbone",   5,  "Conv",    ch(512),        ch(512),     "k=3, s=2, 80→40"),
        ("Backbone",   6,  "C3k2",    ch(512),        ch(512),     f"n={dep(2)}, c3k=True  ← P4"),
        ("Backbone",   7,  "Conv",    ch(512),        ch(1024),    "k=3, s=2, 40→20"),
        ("Backbone",   8,  "C3k2",    ch(1024),       ch(1024),    f"n={dep(2)}, c3k=True"),
        ("Backbone",   9,  "SPPF",    ch(1024),       ch(1024),    "k=5      ← P5"),
        ("Neck  ",    10,  "C2PSA",   ch(1024),       ch(1024),    f"n={dep(2)}"),
        ("Neck  ",    11,  "Upsample","−",            "×2",        "20→40"),
        ("Neck  ",    12,  "Concat",  "11+P4",        "−",         f"{ch(1024)+ch(512)}"),
        ("Neck  ",    13,  "C3k2",    ch(1024)+ch(512),ch(512),   f"n={dep(2)}, c3k=False"),
        ("Neck  ",    14,  "Upsample","−",            "×2",        "40→80"),
        ("Neck  ",    15,  "Concat",  "14+P3",        "−",         f"{ch(512)+ch(512)}"),
        ("Neck  ",    16,  "C3k2",    ch(512)+ch(512),ch(256),    f"n={dep(2)}, c3k=False ← n3"),
        ("Neck  ",    17,  "Conv",    ch(256),        ch(256),     "k=3, s=2, 80→40"),
        ("Neck  ",    18,  "Concat",  "17+13",        "−",         f"{ch(256)+ch(512)}"),
        ("Neck  ",    19,  "C3k2",    ch(256)+ch(512),ch(512),    f"n={dep(2)}, c3k=False ← n4"),
        ("Neck  ",    20,  "Conv",    ch(512),        ch(512),     "k=3, s=2, 40→20"),
        ("Neck  ",    21,  "Concat",  "20+10",        "−",         f"{ch(512)+ch(1024)}"),
        ("Neck  ",    22,  "C3k2",    ch(512)+ch(1024),ch(1024),  f"n={dep(2)}, c3k=True  ← n5"),
        ("Head  ",    23,  "Detect",  f"{ch(256)},{ch(512)},{ch(1024)}", "−", "nc=1"),
    ]
    
    print("=" * 85)
    print(f"  Layer Breakdown — {model_name}")
    print("=" * 85)
    print(f"  {'Section':<12} {'#':<6} {'Type':<12} {'C_in':<20} {'C_out':<12} {'Notes'}")
    print("-" * 85)
    
    prev_section = ""
    for section, num, ltype, cin, cout, notes in layers:
        if section != prev_section and prev_section != "":
            print("-" * 85)
        print(f"  {section:<12} {num:<6} {ltype:<12} {str(cin):<20} {str(cout):<12} {notes}")
        prev_section = section
    
    print("=" * 85)

layer_breakdown("yolo11n")

  Layer Breakdown — yolo11n
  Section      #      Type         C_in                 C_out        Notes
-------------------------------------------------------------------------------------
  Backbone     0      Conv         3                    16           k=3, s=2, 640→320
  Backbone     1      Conv         16                   32           k=3, s=2, 320→160
  Backbone     2      C3k2         32                   64           n=1, c3k=False
  Backbone     3      Conv         64                   64           k=3, s=2, 160→80
  Backbone     4      C3k2         64                   128          n=1, c3k=False ← P3
  Backbone     5      Conv         128                  128          k=3, s=2, 80→40
  Backbone     6      C3k2         128                  128          n=1, c3k=True  ← P4
  Backbone     7      Conv         128                  256          k=3, s=2, 40→20
  Backbone     8      C3k2         256                  256          n=1, c3k=True
  Backbone     9      SPPF         2

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

model = YOLOv11("yolo11n", nc=1).to(device)
dummy = torch.zeros(1, 3, 640, 640).to(device)

model.eval()
with torch.no_grad():
    out = model(dummy)

print(f"Model on  : {next(model.parameters()).device}")
print(f"Input on  : {dummy.device}")
print(f"Output    : {out.shape}")
print(f"Output on : {out.device}")

Device : cpu
Model on  : cpu
Input on  : cpu
Output    : torch.Size([1, 5, 8400])
Output on : cpu
